# 🎙️ YouTube Article Generator: Complete Multi-Agent Pipeline with Human-in-the-Loop (HITL)

## Complete Workflow Architecture:
**Agent 1 (Analyst)** ➔ **Agent 2 (Writer)** ➔ **Agent 3 (Reviewer)** ➔ **Human Review (HITL)** ➔ **Final Approved Article**

### Pipeline Overview:
1. **Transcript Extraction & Chunking**: Extracts subtitles from YouTube and splits into time-based slices.
2. **Agent 1 (YouTube Content Analyst)**: Understands core topic, extracts key points, and produces a structured article outline.
3. **Agent 2 (YouTube Article Writer)**: Transforms transcript and outline into an engaging, complete Markdown article.
4. **Agent 3 (YouTube Article Editorial Reviewer)**: Audits the article against the transcript ground truth for hallucinations, timestamp errors, and missing concepts.
5. **Human-in-the-Loop Review Stage**: Allows human editors to inspect feedback side-by-side, accept/reject specific suggestions, edit text directly, trigger Agent 2 revisions, and issue **Final Publication Approval**.

In [1]:
# Required packages
# !pip install youtube-transcript-api langchain langchain-openai pydantic python-dotenv ipywidgets

In [2]:
import os
import re
import json
import time
from typing import List, Dict, Optional, Any
from urllib.parse import urlparse, parse_qs

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from youtube_transcript_api import YouTubeTranscriptApi

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

try:
    from IPython.display import display, HTML
except ImportError:
    display = print
    HTML = lambda x: x

load_dotenv()

True

## 1. Transcript Extraction
Extracts YouTube Video ID and retrieves timestamped subtitles.

In [3]:
def extract_video_id(url: str) -> str:
    """Extract video ID from standard or shortened YouTube URLs."""
    parsed_url = urlparse(url)

    if parsed_url.hostname in ["www.youtube.com", "youtube.com"]:
        return parse_qs(parsed_url.query)["v"][0]

    if parsed_url.hostname == "youtu.be":
        return parsed_url.path.strip("/")

    raise ValueError("Invalid YouTube URL")


def get_transcript(url: str, retries: int = 3) -> str:
    """Fetch transcript from YouTube and format with timestamps."""
    video_id = extract_video_id(url)
    api = YouTubeTranscriptApi()
    last_error = None

    for attempt in range(retries):
        try:
            if hasattr(api, "fetch"):
                transcript = api.fetch(video_id)
            else:
                transcript = YouTubeTranscriptApi.get_transcript(video_id)
            break
        except Exception as e:
            last_error = e
            print(f"Transcript fetch failed (attempt {attempt + 1}/{retries}): {e}", flush=True)
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    else:
        raise RuntimeError(
            f"Could not retrieve transcript for YouTube video {video_id} after {retries} attempts."
        ) from last_error

    text = []
    for entry in transcript:
        start_time = getattr(entry, "start", None) if hasattr(entry, "start") else entry.get("start", 0)
        content = getattr(entry, "text", None) if hasattr(entry, "text") else entry.get("text", "")
        
        timestamp = int(start_time)
        minutes = timestamp // 60
        seconds = timestamp % 60
        formatted_time = f"{minutes:02d}:{seconds:02d}"
        text.append(f"[{formatted_time}] {content}")

    return "\n".join(text)

## 2. Transcript Chunking
Splits formatted transcripts into time slices (e.g. 5-minute / 300-second windows).

In [4]:
def format_timestamp(seconds: int) -> str:
    """Format seconds into MM:SS string."""
    minutes = seconds // 60
    remaining_seconds = seconds % 60
    return f"{minutes:02d}:{remaining_seconds:02d}"


def chunk_transcript(
    transcript: str,
    chunk_duration: int = 300,
) -> list[dict]:
    """Chunk transcript text into time-based segment buckets."""
    lines = transcript.splitlines()
    chunks = {}
    timestamp_pattern = re.compile(r"\[(\d+):(\d{2})\]\s*(.*)")

    for line in lines:
        match = timestamp_pattern.match(line)
        if not match:
            continue

        minutes = int(match.group(1))
        seconds = int(match.group(2))
        text = match.group(3).strip()

        timestamp_seconds = minutes * 60 + seconds
        chunk_id = timestamp_seconds // chunk_duration

        if chunk_id not in chunks:
            chunks[chunk_id] = []

        chunks[chunk_id].append(
            {
                "timestamp": format_timestamp(timestamp_seconds),
                "text": text,
            }
        )

    result = []
    for chunk_id in sorted(chunks):
        start_seconds = chunk_id * chunk_duration
        end_seconds = (chunk_id + 1) * chunk_duration
        segments = chunks[chunk_id]

        result.append(
            {
                "chunk_id": chunk_id + 1,
                "start_timestamp": format_timestamp(start_seconds),
                "end_timestamp": format_timestamp(end_seconds),
                "segments": segments,
            }
        )

    return result

## 3. Agent 1: YouTube Content Analyst
Analyzes the transcript chunks and creates a structured outline for Agent 2.

In [5]:
class SectionOutline(BaseModel):
    section_heading: str = Field(description="Clear, engaging heading for this section")
    key_points: List[str] = Field(description="Main points, arguments, or concepts to cover in this section")
    relevant_timestamps: List[str] = Field(description="Timestamps from the transcript referenced in this section")
    examples_and_details: List[str] = Field(default_factory=list, description="Specific examples, commands, or details to include")

class ArticleOutline(BaseModel):
    article_title: str = Field(description="Working title for the article")
    main_topic: str = Field(description="Central subject and thesis of the video")
    target_audience: str = Field(description="Intended audience for the article")
    key_takeaways: List[str] = Field(description="3-5 core takeaways that reader should learn")
    sections: List[SectionOutline] = Field(description="Logical, ordered sections of the article")
    uncertainties: List[str] = Field(default_factory=list, description="Any points that were ambiguous or unclear in the transcript")

In [6]:
AGENT_1_SYSTEM_PROMPT = """You are Agent 1: The YouTube Content Analyst.

Your job is to read all transcript chunks from a YouTube video and create a structured, publication-ready article outline for the writer agent.

GUIDELINES:
1. Understand the core topic and formulate an appealing, informative article title.
2. Determine the target audience and primary takeaways.
3. Group the material logically by theme into coherent sections. Do NOT create one section per chunk; organize by topic.
4. Remove fluff: ignore channel intros, subscription requests, greetings, and filler chatter.
5. Preserve original timestamps for every section so the writer can ground every point.
6. Retain important examples, commands, analogies, and technical details.
7. Note any uncertainties or ambiguous statements so the writer does not make false assumptions.
8. DO NOT write the article body yet. Output ONLY the structured outline.
"""

# Shared LLM Configuration with 45s timeout to prevent hanging
llm = ChatOpenAI(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT") or os.getenv("OPENAI_MODEL_NAME") or "gpt-4o",
    openai_api_base=os.getenv("OPENAI_BASE_URL") or os.getenv("AZURE_OPENAI_ENDPOINT"),
    openai_api_key=os.getenv("OPENAI_API_KEY") or os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0.1,
    timeout=45,
    max_retries=1,
)

# Build Agent 1
agent_1 = create_agent(
    model=llm,
    system_prompt=AGENT_1_SYSTEM_PROMPT,
    response_format=ArticleOutline,
)


def generate_article_outline(chunks: list[dict], agent=agent_1) -> ArticleOutline:
    """Takes transcript chunks and executes Agent 1 to generate the article outline."""
    formatted_blocks = []
    for chunk in chunks:
        header = f"--- Chunk {chunk.get('chunk_id')} ({chunk.get('start_timestamp')} to {chunk.get('end_timestamp')}) ---"
        lines = [f"[{seg.get('timestamp')}] {seg.get('text')}" for seg in chunk.get("segments", [])]
        formatted_blocks.append(header + "\n" + "\n".join(lines))

    prompt_content = f"""Here are the transcript chunks from the video:

""" + "\n\n".join(formatted_blocks) + """

Analyze these chunks and generate the structured article outline.
"""

    response = agent.invoke({
        "messages": [{"role": "user", "content": prompt_content}]
    })

    if "structured_response" in response:
        return response["structured_response"]
    elif isinstance(response.get("messages", [])[-1].content, str):
        return ArticleOutline.model_validate_json(response["messages"][-1].content)
    else:
        return response

## 4. Agent 2: YouTube Article Writer (Drafting & Revision)
Writes a complete, readable Markdown article, and provides revision capability based on Human-in-the-Loop decisions.

In [7]:
class ArticleOutput(BaseModel):
    title: str = Field(description="Final polished article title")
    markdown_article: str = Field(
        description="The complete, publication-ready article in Markdown format (including Title, Introduction, Sections with timestamps and code blocks, Conclusion, and YouTube link)"
    )
    word_count: int = Field(description="Estimated word count of the article")


AGENT_2_SYSTEM_PROMPT = """You are Agent 2: The YouTube Article Writer.

Your mission is to write a complete, engaging, publication-ready article based on the original YouTube transcript and the structured outline created by Agent 1.

RESPONSIBILITIES:
1. Follow the structure and section order defined in the article outline from Agent 1.
2. Write a comprehensive article featuring:
   - An engaging Title (# Title)
   - A well-crafted Introduction setting the context
   - Fully developed main sections with appropriate subheadings (## and ###)
   - Concrete code blocks (```bash or ```dockerfile) for all commands and technical examples
   - A dedicated section at the end titled '## Uncertainties and Points to Verify' explicitly covering any ambiguous points flagged by Agent 1 (e.g. uncertain release dates or versions)
   - A concise, thoughtful Conclusion summarizing key insights
   - The original YouTube video link referenced at the end
3. Convert spoken transcript explanations into clear, professional written prose.
4. Strip conversational filler words, greetings, like/subscribe requests, and repetitions.
5. Ground every technical explanation, command, and fact strictly in the transcript.
6. Embed original timestamps (e.g., `[02:15]`) within the relevant sections so readers can trace key concepts to the original video.
7. Ensure the article is entirely self-contained and clear to a reader who has not watched the video.

STRICT RULES:
- DO NOT invent facts, commands, libraries, or versions not present in the transcript.
- ALWAYS include '## Uncertainties and Points to Verify' before the conclusion if Agent 1 flagged any uncertainties.
- DO NOT copy the transcript verbatim; rephrase into smooth, authoritative writing.
- DO NOT output internal reasoning, apologies, or meta-comments.
- Output ONLY the final validated ArticleOutput containing the complete Markdown text.
"""

# Build Agent 2
agent_2 = create_agent(
    model=llm,
    system_prompt=AGENT_2_SYSTEM_PROMPT,
    response_format=ArticleOutput,
)


def write_article(
    video_title: str,
    chunks: list[dict],
    outline: ArticleOutline,
    video_url: str = "https://www.youtube.com/watch?v=sample",
    article_style: str = "technical, clear, and authoritative",
    target_audience: Optional[str] = None,
    video_description: Optional[str] = "",
    agent=agent_2,
) -> ArticleOutput:
    """Invokes Agent 2 to author the full Markdown article."""
    formatted_transcript = []
    for chunk in chunks:
        header = f"--- Chunk {chunk.get('chunk_id')} ({chunk.get('start_timestamp')} to {chunk.get('end_timestamp')}) ---"
        lines = [f"[{seg.get('timestamp')}] {seg.get('text')}" for seg in chunk.get("segments", [])]
        formatted_transcript.append(header + "\n" + "\n".join(lines))

    full_transcript_str = "\n\n".join(formatted_transcript)
    audience = target_audience or outline.target_audience

    user_prompt = f"""Write the full article using the following inputs:

=== VIDEO INFORMATION ===
Video Title: {video_title}
Video URL: {video_url}
Video Description: {video_description or 'N/A'}
Target Audience: {audience}
Desired Style: {article_style}

=== ARTICLE OUTLINE FROM AGENT 1 ===
{outline.model_dump_json(indent=2)}

=== ORIGINAL TRANSCRIPT CHUNKS (GROUND TRUTH) ===
{full_transcript_str}

Produce the complete, polished Markdown article following the outline and instructions.
"""

    response = agent.invoke({
        "messages": [{"role": "user", "content": user_prompt}]
    })

    if "structured_response" in response:
        return response["structured_response"]
    elif isinstance(response.get("messages", [])[-1].content, str):
        return ArticleOutput.model_validate_json(response["messages"][-1].content)
    else:
        return response


# Agent 2 Revision Capability
AGENT_2_REVISE_PROMPT = """You are Agent 2: The YouTube Article Writer (Revision Mode).

Your mission is to revise the previous draft of the article based on Human Editorial Decisions and Agent 3 Review Feedback.

HUMAN-IN-THE-LOOP PRINCIPLES:
1. Human decisions take absolute priority over Agent 3's feedback.
2. If the human accepted an issue or requested a change, implement that modification cleanly.
3. If the human rejected an issue (e.g. electing to keep a technically accurate context explanation even if not in the video), keep that explanation intact!
4. Clearly distinguish transcript-supported content from additional technical context.
5. Ensure uncertain facts remain in the '## Uncertainties and Points to Verify' section.
6. Return the updated, publication-ready ArticleOutput in Markdown format.
"""

agent_2_revision = create_agent(
    model=llm,
    system_prompt=AGENT_2_REVISE_PROMPT,
    response_format=ArticleOutput,
)


def revise_article(
    previous_article_markdown: str,
    human_instructions: str,
    accepted_feedback: List[str],
    chunks: list[dict],
    outline: ArticleOutline,
    agent=agent_2_revision,
) -> ArticleOutput:
    """Revises the article incorporating accepted human editorial feedback."""
    formatted_transcript = []
    for chunk in chunks:
        header = f"--- Chunk {chunk.get('chunk_id')} ({chunk.get('start_timestamp')} to {chunk.get('end_timestamp')}) ---"
        lines = [f"[{seg.get('timestamp')}] {seg.get('text')}" for seg in chunk.get("segments", [])]
        formatted_transcript.append(header + "\n" + "\n".join(lines))

    full_transcript_str = "\n\n".join(formatted_transcript)

    revision_prompt = f"""Revise the article according to the human editor's instructions:

=== PREVIOUS ARTICLE DRAFT ===
{previous_article_markdown}

=== ACCEPTED EDITORIAL ISSUES & INSTRUCTIONS ===
{json.dumps(accepted_feedback, indent=2)}

=== DIRECT HUMAN INSTRUCTIONS ===
{human_instructions or 'Apply all accepted editorial corrections cleanly.'}

=== ORIGINAL TRANSCRIPT CHUNKS (GROUND TRUTH) ===
{full_transcript_str}

Produce the revised, polished Markdown article.
"""

    response = agent.invoke({
        "messages": [{"role": "user", "content": revision_prompt}]
    })

    if "structured_response" in response:
        return response["structured_response"]
    elif isinstance(response.get("messages", [])[-1].content, str):
        return ArticleOutput.model_validate_json(response["messages"][-1].content)
    else:
        return response

## 5. Agent 3: YouTube Article Editorial Reviewer
Audits the article against the transcript ground truth and Agent 1's outline.

In [8]:
class ReviewIssue(BaseModel):
    type: str = Field(
        description="Category of issue: 'unsupported_claim', 'technical_distortion', 'hallucinated_fact', or 'structure_issue'"
    )
    section: str = Field(description="Section heading where the issue was detected")
    issue: str = Field(description="Detailed description of the issue")
    timestamp: Optional[str] = Field(default=None, description="Associated timestamp from article or transcript (e.g. '01:15')")
    severity: str = Field(description="Severity level: 'high', 'medium', or 'low'")


class EditorialReviewOutput(BaseModel):
    approved: bool = Field(
        description="True if the article is sufficiently accurate, grounded, complete, and readable; False if it has critical unsupported claims, distorted explanations, or missing core concepts"
    )
    overall_feedback: str = Field(description="Concise summary assessment of the article quality and grounding")
    issues: List[ReviewIssue] = Field(default_factory=list, description="List of identified issues")
    missing_content: List[str] = Field(
        default_factory=list,
        description="Essential concepts, examples, or takeaways from Agent 1 outline missing from the article"
    )
    timestamp_errors: List[str] = Field(
        default_factory=list,
        description="Timestamps in the article that do not exist in the transcript or fail to support the claimed point"
    )
    uncertainties_missing: List[str] = Field(
        default_factory=list,
        description="Uncertain facts identified by Agent 1 that were omitted from the 'Uncertainties and Points to Verify' section"
    )
    writing_issues: List[str] = Field(
        default_factory=list,
        description="Repetitive, irrelevant, or poorly structured sections"
    )
    revision_instructions: List[str] = Field(
        default_factory=list,
        description="Clear, actionable revision instructions for Agent 2 if changes are required"
    )


AGENT_3_SYSTEM_PROMPT = """You are Agent 3: The YouTube Article Editorial Reviewer.

Your purpose is quality control. You review the article written by Agent 2 against the original transcript and Agent 1's structured outline.
You are an auditor and editor, NOT an article writer. Do not rewrite the article.

EVALUATION CRITERIA:
1. Transcript Grounding (Source of Truth):
   - Check whether claims in the article are supported by the original transcript.
   - Identify facts, commands, libraries, or explanations added without transcript support.
   - Distinguish between unsupported claims and technically incorrect claims. A statement might be true in general computing, but if the speaker never said it, note it as an unsupported claim without treating general knowledge as proof that the speaker mentioned it.
2. Outline & Concept Completeness:
   - Verify that all core concepts, sections, and important examples from Agent 1's outline are covered.
3. Timestamp Verification:
   - Verify that timestamps referenced in the article actually exist in the transcript and support the claimed point.
4. Technical Accuracy:
   - Check whether technical explanations have been distorted into inaccuracies.
5. Structure and Flow:
   - Identify repetitive or poorly structured prose.
6. Uncertainties Check:
   - Check whether uncertain facts flagged by Agent 1 are included in the section titled 'Uncertainties and Points to Verify'.

DECISION GUIDELINES:
- Set approved = True if the article is sufficiently accurate, grounded, complete, and readable. Minor stylistic issues or small helpful clarifications should NOT cause rejection.
- Set approved = False only if it contains critical factual hallucinations, severe distortions, or omits major sections.
- Always provide clear, encouraging feedback when approving, or actionable revision instructions when revision is recommended.
"""

# Build Agent 3
agent_3 = create_agent(
    model=llm,
    system_prompt=AGENT_3_SYSTEM_PROMPT,
    response_format=EditorialReviewOutput,
)


def review_article(
    chunks: list[dict],
    outline: ArticleOutline,
    article: ArticleOutput,
    agent=agent_3,
) -> EditorialReviewOutput:
    """
    Invokes Agent 3 to review and audit the generated article.
    """
    formatted_transcript = []
    for chunk in chunks:
        header = f"--- Chunk {chunk.get('chunk_id')} ({chunk.get('start_timestamp')} to {chunk.get('end_timestamp')}) ---"
        lines = [f"[{seg.get('timestamp')}] {seg.get('text')}" for seg in chunk.get("segments", [])]
        formatted_transcript.append(header + "\n" + "\n".join(lines))

    full_transcript_str = "\n\n".join(formatted_transcript)

    audit_prompt = f"""Please conduct an editorial review of the following article against the outline and original transcript chunks:

=== ORIGINAL TRANSCRIPT CHUNKS (GROUND TRUTH) ===
{full_transcript_str}

=== AGENT 1 ARTICLE OUTLINE ===
{outline.model_dump_json(indent=2)}

=== AGENT 2 GENERATED ARTICLE ===
Title: {article.title}

{article.markdown_article}

Perform all checks and return your structured evaluation in the required JSON format.
"""

    response = agent.invoke({
        "messages": [{"role": "user", "content": audit_prompt}]
    })

    if "structured_response" in response:
        return response["structured_response"]
    elif isinstance(response.get("messages", [])[-1].content, str):
        return EditorialReviewOutput.model_validate_json(response["messages"][-1].content)
    else:
        return response

## 6. Human-in-the-Loop (HITL) Review Stage

The **Human-in-the-Loop Review Stage** sits between Agent 3 and publication.

### Core Principles:
1. **Human Authority**: Human decisions supersede Agent 3. The editor can accept, modify, or reject any recommendation.
2. **Nuanced Evaluation**: An explanation is not rejected merely because it was not in the video; technically sound background context can be accepted at the editor's discretion.
3. **Explicit Provenance**: Clearly distinguishes **[TRANSCRIPT-SUPPORTED]** claims from **[ADDITIONAL AI EXPLANATION]** and **[UNCERTAINTIES]**.
4. **Direct & Guided Editing**: Allows direct markdown edits OR automated Agent 2 revisions using selected feedback.
5. **Audit History**: Preserves previous drafts and review results for side-by-side comparison.
6. **Prominent Approval Message**: Issues a visible, unambiguous approval certificate upon publication sign-off.

In [9]:
class HumanReviewSession:
    """
    Manages the Human-in-the-Loop Review Stage.
    Tracks versions, user decisions on issues, direct edits, and revision triggers.
    """
    def __init__(
        self,
        initial_article: ArticleOutput,
        review_report: EditorialReviewOutput,
        chunks: list[dict],
        outline: ArticleOutline,
    ):
        self.chunks = chunks
        self.outline = outline
        self.current_article_markdown = initial_article.markdown_article
        self.current_title = initial_article.title
        self.review_report = review_report
        self.history: List[Dict[str, Any]] = [
            {
                "version": 1,
                "article_markdown": initial_article.markdown_article,
                "review": review_report,
            }
        ]
        self.issue_decisions: Dict[int, Dict[str, str]] = {}
        self.is_final_approved: bool = False
        self.final_article_markdown: Optional[str] = None

    def display_editorial_dashboard(self):
        """Displays side-by-side article preview and categorized feedback."""
        print("=" * 80, flush=True)
        print("               🔍 HUMAN-IN-THE-LOOP EDITORIAL REVIEW DASHBOARD", flush=True)
        print("=" * 80, flush=True)
        print(f"Current Title: {self.current_title}", flush=True)
        status = '✅ APPROVED FOR PUBLICATION' if self.is_final_approved else '⏳ PENDING HUMAN DECISION'
        print(f"Publication Status: {status}", flush=True)
        print(f"Agent 3 Recommendation: {'APPROVED' if self.review_report.approved else 'REVISION RECOMMENDED'}", flush=True)
        print(f"Agent 3 Feedback: {self.review_report.overall_feedback}", flush=True)
        print("-" * 80, flush=True)

        # Review Issues
        print("\n📋 AGENT 3 SUGGESTIONS & ISSUES:", flush=True)
        if not self.review_report.issues:
            print("  (No issues detected by Agent 3)", flush=True)
        for idx, issue in enumerate(self.review_report.issues):
            decision = self.issue_decisions.get(idx, {}).get("decision", "UNREVIEWED")
            badge = f"[{decision}]"
            print(f"  [{idx}] {badge} [{issue.severity.upper()}] ({issue.type}) in '{issue.section}'", flush=True)
            print(f"      Problem: {issue.issue}", flush=True)
            if issue.timestamp:
                print(f"      Timestamp: {issue.timestamp}", flush=True)

        # Missing Outline Content
        if self.review_report.missing_content:
            print("\n🔍 MISSING CONTENT FROM OUTLINE:", flush=True)
            for mc in self.review_report.missing_content:
                print(f"  • {mc}", flush=True)

        # Timestamp Errors
        if self.review_report.timestamp_errors:
            print("\n⏱️ TIMESTAMP ERRORS:", flush=True)
            for te in self.review_report.timestamp_errors:
                print(f"  • {te}", flush=True)

        # Missing Uncertainties
        if self.review_report.uncertainties_missing:
            print("\n❓ UNCERTAINTIES NOT COVERED IN FINAL SECTION:", flush=True)
            for um in self.review_report.uncertainties_missing:
                print(f"  • {um}", flush=True)

        print("=" * 80, flush=True)

    def accept_suggestion(self, issue_index: int, notes: str = ""):
        """Accepts Agent 3's feedback item to be applied in revision."""
        if 0 <= issue_index < len(self.review_report.issues):
            self.issue_decisions[issue_index] = {
                "decision": "ACCEPTED",
                "notes": notes,
                "issue": self.review_report.issues[issue_index].issue,
                "section": self.review_report.issues[issue_index].section,
            }
            print(f"✓ Issue [{issue_index}] marked as ACCEPTED.", flush=True)
        else:
            print(f"Invalid issue index: {issue_index}", flush=True)

    def reject_suggestion(self, issue_index: int, reason: str = "Keep technically accurate explanation"):
        """Rejects Agent 3's feedback item (human override)."""
        if 0 <= issue_index < len(self.review_report.issues):
            self.issue_decisions[issue_index] = {
                "decision": "REJECTED",
                "notes": reason,
                "issue": self.review_report.issues[issue_index].issue,
                "section": self.review_report.issues[issue_index].section,
            }
            print(f"✓ Issue [{issue_index}] marked as REJECTED (Human Override: {reason}).", flush=True)
        else:
            print(f"Invalid issue index: {issue_index}", flush=True)

    def direct_edit_article(self, updated_markdown: str):
        """Allows human editor to edit article markdown directly."""
        self.current_article_markdown = updated_markdown
        self.history.append({
            "version": len(self.history) + 1,
            "article_markdown": updated_markdown,
            "action": "Direct Human Edit",
        })
        print(f"✓ Article directly updated by human editor (Version {len(self.history)}).", flush=True)

    def request_agent_revision(self, custom_instructions: str = "") -> ArticleOutput:
        """Triggers Agent 2 to produce a revised draft applying only accepted suggestions."""
        accepted_items = [
            f"Section '{data['section']}': {data['issue']}" + (f" (Note: {data['notes']})" if data['notes'] else "")
            for data in self.issue_decisions.values()
            if data["decision"] == "ACCEPTED"
        ]

        print("\n🔄 Requesting revision from Agent 2 with accepted human decisions...", flush=True)
        revised = revise_article(
            previous_article_markdown=self.current_article_markdown,
            human_instructions=custom_instructions,
            accepted_feedback=accepted_items,
            chunks=self.chunks,
            outline=self.outline,
        )

        print("🔎 Running Agent 3 quality-audit on revised draft...", flush=True)
        new_review = review_article(self.chunks, self.outline, revised)

        self.current_article_markdown = revised.markdown_article
        self.current_title = revised.title
        self.review_report = new_review
        self.issue_decisions = {}
        self.history.append({
            "version": len(self.history) + 1,
            "article_markdown": revised.markdown_article,
            "review": new_review,
            "action": "Agent 2 Revision",
        })
        print(f"✓ Revision complete! (Now on Draft v{len(self.history)})", flush=True)
        return revised

    def approve_for_publication(self, editor_notes: str = "All checks passed and approved by editor.") -> str:
        """
        Grants final human approval, renders the prominent approval certificate, and returns the final article.
        """
        self.is_final_approved = True
        self.final_article_markdown = self.current_article_markdown

        # 1. Plain Text Approval Message
        approval_banner = f"""
********************************************************************************
🏆 [EDITORIAL APPROVAL CERTIFICATE]
********************************************************************************
Status           : ✅ OFFICIALLY APPROVED FOR PUBLICATION
Article Title    : {self.current_title}
Version Approved : v{len(self.history)} (Final Publication Build)
Editor Sign-off  : {editor_notes}
Agent 3 Audit    : {'APPROVED' if self.review_report.approved else 'OVERRIDDEN BY HUMAN EDITOR (All concerns resolved)'}
Next Step        : Article is ready for CMS / Blog / Markdown deployment!
********************************************************************************
"""
        print(approval_banner, flush=True)

        # 2. Rich HTML Approval Badge (for Jupyter Notebook rendering)
        html_badge = f"""
        <div style="
            border: 2px solid #28a745;
            background: linear-gradient(135deg, #d4edda 0%, #c3e6cb 100%);
            padding: 20px;
            border-radius: 10px;
            margin: 15px 0;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
        ">
            <div style="display: flex; align-items: center;">
                <span style="font-size: 36px; margin-right: 15px;">🎉</span>
                <div>
                    <h2 style="margin: 0; color: #155724; font-size: 22px;">Article Approved for Publication!</h2>
                    <p style="margin: 5px 0 0 0; color: #1e7e34; font-size: 14px;">
                        <b>Title:</b> {self.current_title} | <b>Version:</b> v{len(self.history)} | <b>Status:</b> Ready for Release
                    </p>
                </div>
            </div>
        </div>
        """
        try:
            display(HTML(html_badge))
        except Exception:
            pass

        return self.final_article_markdown

## 7. Interactive Widget (Optional GUI Support)
If `ipywidgets` is installed, this cell provides an interactive UI with live side-by-side editing, accept/reject toggles, and revision buttons directly in the notebook.

In [10]:
def render_interactive_human_review(session: HumanReviewSession):
    """
    Renders an interactive side-by-side GUI in Jupyter if ipywidgets is installed.
    Falls back gracefully to the standard console dashboard if not present.
    """
    try:
        import ipywidgets as widgets
        from IPython.display import display

        # Left panel: Editable Article
        editor_label = widgets.HTML("<h3>📝 Editable Article Preview</h3>")
        editor_area = widgets.Textarea(
            value=session.current_article_markdown,
            layout=widgets.Layout(width="100%", height="480px", font_family="monospace")
        )

        # Right panel: Agent 3 Feedback Cards with Accept/Reject
        feedback_label = widgets.HTML("<h3>🛡️ Agent 3 Review Feedback</h3>")
        feedback_boxes = []
        for idx, issue in enumerate(session.review_report.issues):
            issue_html = widgets.HTML(
                f"<b>[{issue.severity.upper()}] ({issue.type})</b> in <i>{issue.section}</i><br/>"
                f"{issue.issue}<br/>"
                f"<small>Timestamp: {issue.timestamp or 'N/A'}</small>"
            )
            decision_toggle = widgets.ToggleButtons(
                options=["Unreviewed", "Accept", "Reject"],
                value="Unreviewed",
                button_style="",
            )
            def make_handler(i, toggle):
                def on_change(change):
                    if change["new"] == "Accept":
                        session.accept_suggestion(i)
                    elif change["new"] == "Reject":
                        session.reject_suggestion(i)
                return on_change

            decision_toggle.observe(make_handler(idx, decision_toggle), names="value")
            card = widgets.VBox([issue_html, decision_toggle], layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="4px"))
            feedback_boxes.append(card)

        feedback_container = widgets.VBox(feedback_boxes, layout=widgets.Layout(height="480px", overflow_y="auto"))

        # Action Buttons
        btn_save = widgets.Button(description="Save Direct Edits", button_style="info")
        btn_revise = widgets.Button(description="Request Agent 2 Revision", button_style="warning")
        btn_approve = widgets.Button(description="Approve & Finalize", button_style="success")
        status_output = widgets.Output()

        def on_save_clicked(b):
            session.direct_edit_article(editor_area.value)
            with status_output:
                status_output.clear_output()
                print("✓ Direct edits saved successfully!", flush=True)

        def on_revise_clicked(b):
            with status_output:
                status_output.clear_output()
                print("Revising article with Agent 2...", flush=True)
            revised = session.request_agent_revision()
            editor_area.value = revised.markdown_article
            with status_output:
                print(f"✓ Revised article updated in editor (v{len(session.history)})!", flush=True)

        def on_approve_clicked(b):
            with status_output:
                status_output.clear_output()
            session.approve_for_publication()

        btn_save.on_click(on_save_clicked)
        btn_revise.on_click(on_revise_clicked)
        btn_approve.on_click(on_approve_clicked)

        # Layout Assembly
        left_box = widgets.VBox([editor_label, editor_area], layout=widgets.Layout(width="55%"))
        right_box = widgets.VBox([feedback_label, feedback_container], layout=widgets.Layout(width="45%"))
        columns = widgets.HBox([left_box, right_box])
        action_bar = widgets.HBox([btn_save, btn_revise, btn_approve])

        display(widgets.VBox([columns, action_bar, status_output]))
    except ImportError:
        # Graceful fallback
        session.display_editorial_dashboard()
        print("\n(Note: Install ipywidgets for interactive GUI buttons. Programmatic review methods are fully functional above.)", flush=True)

## 8. Full End-to-End Pipeline Execution with Human Approval

Runs the complete workflow:
1. **Agent 1** (Analyst) ➔ Outline
2. **Agent 2** (Writer) ➔ Draft Article
3. **Agent 3** (Editorial Reviewer) ➔ Quality Report
4. **Human Review Stage** ➔ Editorial Decisions, Override/Acceptance, and Revision
5. **Final Approved Article** ➔ Official Approval Certificate & Publication

In [11]:
# Sample Docker tutorial transcript chunks
sample_chunks = [
    {
        "chunk_id": 1,
        "start_timestamp": "00:00",
        "end_timestamp": "05:00",
        "segments": [
            {"timestamp": "00:05", "text": "Hey everyone, welcome back! Please hit subscribe and like the video."}, 
            {"timestamp": "00:20", "text": "Today we're talking about Docker containers vs Virtual Machines."}, 
            {"timestamp": "01:15", "text": "Virtual machines require a hypervisor and a full guest OS, which consumes gigabytes of memory."}, 
            {"timestamp": "03:10", "text": "Docker containers share the host Linux kernel using cgroups and namespaces for process isolation."}, 
            {"timestamp": "04:30", "text": "This means containers boot in milliseconds rather than minutes and solve the classic 'works on my machine' issue."} 
        ]
    },
    {
        "chunk_id": 2,
        "start_timestamp": "05:00",
        "end_timestamp": "10:00",
        "segments": [
            {"timestamp": "05:10", "text": "Let's look at Dockerfiles. You write your instructions to build an image."}, 
            {"timestamp": "06:20", "text": "You build the image using 'docker build -t myapp:latest .'."}, 
            {"timestamp": "07:35", "text": "To run it in detached mode on port 8080, run 'docker run -d -p 8080:80 myapp:latest'."}, 
            {"timestamp": "08:45", "text": "Images are read-only templates, and containers are the runnable instances."}, 
            {"timestamp": "09:30", "text": "You can push and pull them to Docker Hub registry."} 
        ]
    },
    {
        "chunk_id": 3,
        "start_timestamp": "10:00",
        "end_timestamp": "15:00",
        "segments": [
            {"timestamp": "10:15", "text": "For production, always use multi-stage builds to keep image size small."}, 
            {"timestamp": "11:30", "text": "Compile in Golang or Node image, then copy only the static binary into a minimal Alpine or scratch image, cutting size from 1GB to 50MB."}, 
            {"timestamp": "12:45", "text": "For database persistence, always attach named volumes with '-v mydata:/var/lib/postgresql/data'."}, 
            {"timestamp": "13:50", "text": "I think Docker version 25 was released in early 2024, but verify the exact release dates in the docs."}, 
            {"timestamp": "14:35", "text": "Thanks for watching! See you in the next one."} 
        ]
    }
]

# -----------------------------------------------------
# STEP 1: Agent 1 (Analyst)
# -----------------------------------------------------
print("=== 1. AGENT 1: GENERATING OUTLINE ===", flush=True)
outline = generate_article_outline(sample_chunks)
print(f"✓ Outline generated: '{outline.article_title}'\n", flush=True)

# -----------------------------------------------------
# STEP 2: Agent 2 (Writer)
# -----------------------------------------------------
print("=== 2. AGENT 2: WRITING ARTICLE DRAFT ===", flush=True)
article = write_article(
    video_title="Docker Fundamentals: Containers vs VMs and Production Best Practices",
    chunks=sample_chunks,
    outline=outline,
    video_url="https://www.youtube.com/watch?v=docker-example-123",
)
print(f"✓ Draft authored ({article.word_count} words)\n", flush=True)

# -----------------------------------------------------
# STEP 3: Agent 3 (Editorial Reviewer)
# -----------------------------------------------------
print("=== 3. AGENT 3: CONDUCTING EDITORIAL AUDIT ===", flush=True)
review = review_article(sample_chunks, outline, article)
print(f"✓ Audit complete! Agent 3 Decision: {'✅ APPROVED' if review.approved else '⚠️ REVISION RECOMMENDED'}", flush=True)
print(f"✓ Feedback: {review.overall_feedback}\n", flush=True)

# -----------------------------------------------------
# STEP 4: HUMAN-IN-THE-LOOP (HITL) REVIEW STAGE
# -----------------------------------------------------
print("=== 4. HUMAN-IN-THE-LOOP REVIEW STAGE ===", flush=True)
hitl_session = HumanReviewSession(
    initial_article=article,
    review_report=review,
    chunks=sample_chunks,
    outline=outline,
)

# Display the editorial dashboard
hitl_session.display_editorial_dashboard()

# Optional GUI launcher if ipywidgets is present
render_interactive_human_review(hitl_session)

# -----------------------------------------------------
# STEP 5: EDITORIAL DECISIONS & FINAL APPROVAL
# -----------------------------------------------------
# Human editor overrides or accepts suggestions:
if hitl_session.review_report.issues:
    for i in range(len(hitl_session.review_report.issues)):
        # Human decides to accept or override:
        hitl_session.reject_suggestion(i, reason="Verified accurate technical context for readers")

# Final Sign-Off & Official Approval Message
final_article_markdown = hitl_session.approve_for_publication(
    editor_notes="Article verified against transcript, all technical points validated, and approved for publication."
)

print("--- PUBLICATION ARTICLE PREVIEW (First 500 characters) ---", flush=True)
print(final_article_markdown[:500] + "\n\n[... Full article ready for deployment ...]", flush=True)

=== 1. AGENT 1: GENERATING OUTLINE ===
✓ Outline generated: 'Docker Containers vs. Virtual Machines: How Docker Works, Core Commands, and Production Best Practices'

=== 2. AGENT 2: WRITING ARTICLE DRAFT ===
✓ Draft authored (1429 words)

=== 3. AGENT 3: CONDUCTING EDITORIAL AUDIT ===
✓ Audit complete! Agent 3 Decision: ✅ APPROVED
✓ Feedback: The article is strong overall: it is well structured, closely grounded in the transcript, covers all major sections from the outline, includes the required uncertainty section, and uses timestamps that align with the cited points. There are a few minor places where the article adds reasonable explanatory language not stated explicitly in the transcript, but these do not materially distort the source or justify rejection.

=== 4. HUMAN-IN-THE-LOOP REVIEW STAGE ===
               🔍 HUMAN-IN-THE-LOOP EDITORIAL REVIEW DASHBOARD
Current Title: Docker Containers vs. Virtual Machines: How Docker Works, Core Commands, and Production Best Practices
Publica

✓ Issue [0] marked as REJECTED (Human Override: Verified accurate technical context for readers).
✓ Issue [1] marked as REJECTED (Human Override: Verified accurate technical context for readers).
✓ Issue [2] marked as REJECTED (Human Override: Verified accurate technical context for readers).
✓ Issue [3] marked as REJECTED (Human Override: Verified accurate technical context for readers).
✓ Issue [4] marked as REJECTED (Human Override: Verified accurate technical context for readers).

********************************************************************************
🏆 [EDITORIAL APPROVAL CERTIFICATE]
********************************************************************************
Status           : ✅ OFFICIALLY APPROVED FOR PUBLICATION
Article Title    : Docker Containers vs. Virtual Machines: How Docker Works, Core Commands, and Production Best Practices
Version Approved : v1 (Final Publication Build)
Editor Sign-off  : Article verified against transcript, all technical points validate

--- PUBLICATION ARTICLE PREVIEW (First 500 characters) ---
# Docker Containers vs. Virtual Machines: How Docker Works, Core Commands, and Production Best Practices

Docker is often the first container technology developers encounter when they begin exploring modern application packaging and deployment. To use it effectively, it helps to understand not only the commands, but also the underlying model: how containers differ from virtual machines, why they start so quickly, how Docker images are built, and which production practices matter most.

This arti

[... Full article ready for deployment ...]
